# Portable Kaggle build job

This private notebook version contains the source snapshot submitted by the host. Kaggle executes exactly one project Make target in an isolated batch VM and saves its log. Artifact collection is a separate explicit host request.

## Job setup

Restore the submitted source snapshot in temporary storage and define the shared Make runner.

In [ ]:
from pathlib import Path, PurePosixPath
import base64
import io
import inspect
import json
import shutil
import subprocess
import tarfile

SOURCE_ARCHIVE_B64 = '__SOURCE_ARCHIVE_B64__'
REQUESTED_TARGET = base64.urlsafe_b64decode('__REQUESTED_TARGET_B64__').decode()
JOBS = '__JOBS__'
MAKEFILE = base64.urlsafe_b64decode('__MAKEFILE_B64__').decode()
PROJECT_ARGUMENTS = json.loads(base64.urlsafe_b64decode('__PROJECT_ARGUMENTS_B64__').decode())
COLLECT_DIR_B64 = '__COLLECT_DIR_B64__'
COLLECT_DIR = base64.urlsafe_b64decode(COLLECT_DIR_B64).decode() if COLLECT_DIR_B64 else None

ROOT = Path('/tmp/cloud-build')
SOURCE = ROOT / 'src'
RUN_LOG = Path('/kaggle/working/cloud-build.log')
if not REQUESTED_TARGET or '\n' in REQUESTED_TARGET or not isinstance(PROJECT_ARGUMENTS, list) or not all(isinstance(value, str) for value in PROJECT_ARGUMENTS):
    raise ValueError('invalid cloudmake target or project arguments')
if COLLECT_DIR is not None:
    collect_relative = PurePosixPath(COLLECT_DIR)
    if not COLLECT_DIR or collect_relative.is_absolute() or '..' in collect_relative.parts:
        raise ValueError('invalid cloudmake collection directory')

missing_commands = [name for name in ('make', 'tar') if shutil.which(name) is None]
if missing_commands:
    raise RuntimeError(f'missing required remote command(s): {missing_commands}')

def within(path, root):
    return path == root or root in path.parents

def extract_safely(archive, destination):
    root = destination.resolve()
    for member in archive.getmembers():
        member_path = PurePosixPath(member.name)
        if not member.name or member_path.is_absolute() or '..' in member_path.parts:
            raise ValueError(f'unsafe archive member: {member.name}')
        target = (destination / Path(*member_path.parts)).resolve()
        if not within(target, root):
            raise ValueError(f'unsafe archive member: {member.name}')
        if member.ischr() or member.isblk() or member.isfifo() or member.isdev():
            raise ValueError(f'unsafe archive member type: {member.name}')
        if member.issym() or member.islnk():
            link_path = PurePosixPath(member.linkname)
            link_base = target.parent if member.issym() else root
            link_target = (link_base / Path(*link_path.parts)).resolve()
            if link_path.is_absolute() or not within(link_target, root):
                raise ValueError(f'unsafe archive link: {member.name}')
        options = ({'filter': 'data'} if 'filter' in inspect.signature(tarfile.TarFile.extract).parameters else {})
        archive.extract(member, destination, **options)

shutil.rmtree(ROOT, ignore_errors=True)
SOURCE.mkdir(parents=True)
archive_bytes = base64.b64decode(SOURCE_ARCHIVE_B64)
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode='r:gz') as archive:
    extract_safely(archive, SOURCE)

print(f'Workspace: {SOURCE}')
print(f'Requested target: {REQUESTED_TARGET}')
print(f'Parallel jobs: {JOBS}')

def run_make(target):
    command = [
        'make', '-C', str(SOURCE), '-f', MAKEFILE,
        *PROJECT_ARGUMENTS,
        f'-j{JOBS}', '--', target,
    ]
    print('+', ' '.join(command))
    completed = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(completed.stdout, end='')
    RUN_LOG.write_text(completed.stdout)
    completed.check_returncode()

## Project target

Pass the requested name to the project Makefile without a cloudmake target whitelist.

In [ ]:
run_make(REQUESTED_TARGET)

## Optional artifact collection

Only the host `--collect DIR TARGET` operation requests an archive. The target name itself receives no special treatment.

In [ ]:
if COLLECT_DIR is not None:
    source_root = SOURCE.resolve()
    collect_path = (SOURCE / Path(*collect_relative.parts)).resolve()
    if collect_path != source_root and source_root not in collect_path.parents:
        raise ValueError('collection directory escapes the project')
    if not collect_path.is_dir():
        raise FileNotFoundError(f'collection directory does not exist: {COLLECT_DIR}')
    artifact = Path('/kaggle/working/artifacts.tar.gz')
    artifact.unlink(missing_ok=True)
    shutil.make_archive(str(artifact)[:-7], 'gztar', root_dir=collect_path)
    print(f'Artifacts ready at {artifact}')